## ASVSpoof LA19 

In [ ]:
# %%
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

from pathlib import Path
import pandas as pd
import random

# ===== Paths & Seed =====
SRC = Path("./LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.eval.gi.trl.txt")
DST = Path("./PeerJ/protocol/ASVspoof2019.csv")
SEED = 251  # reproducibility
random.seed(SEED)

DST.parent.mkdir(parents=True, exist_ok=True)

# ===== Load Data =====
rows = []
with open(SRC, "r") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) < 4:
            continue
        rows.append(parts)

df = pd.DataFrame(rows, columns=["speaker_id", "utt_id", "label", "trial_type"])

print(f"✅ Loaded {len(df)} rows")

# ===== Split by trial_type =====
df_target = df[df["trial_type"] == "target"]
df_non = df[df["trial_type"] == "nontarget"]
df_spoof = df[df["trial_type"] == "spoof"]

n_target = len(df_target)
n_non = len(df_non)
n_spoof = len(df_spoof)

print(f"\nCounts by trial_type:")
print(f"  target    : {n_target}")
print(f"  nontarget : {n_non}")
print(f"  spoof     : {n_spoof}")

# ===== Balance 'nontarget' and 'spoof' by downsampling to the smaller size =====
balance_n = min(n_non, n_spoof)

if n_non > balance_n:
    df_non_bal = df_non.sample(n=balance_n, random_state=SEED)
else:
    df_non_bal = df_non.copy()

if n_spoof > balance_n:
    df_spoof_bal = df_spoof.sample(n=balance_n, random_state=SEED)
else:
    df_spoof_bal = df_spoof.copy()

print(f"\nBalanced counts (based on min of nontarget/spoof = {balance_n}):")
print(f"  nontarget -> {len(df_non_bal)}")
print(f"  spoof     -> {len(df_spoof_bal)}")
print(f"  target    -> {len(df_target)} (unchanged)")

# ===== Concatenate and shuffle =====
df_final = pd.concat([df_target, df_non_bal, df_spoof_bal], axis=0)
df_final = df_final.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print(f"\nFinal total rows: {len(df_final)}")
print("Final trial_type counts:")
print(df_final["trial_type"].value_counts())

# ===== Save =====
with open(DST, "w") as f:
    for _, row in df_final.iterrows():
        f.write(f"{row['speaker_id']} {row['utt_id']} {row['label']} {row['trial_type']}\n")

print(f"\n💾 Saved balanced protocol → {DST}")


## ASVSpoof 5

In [ ]:
# %%
import pandas as pd
from pathlib import Path
import random
import numpy as np

# ===== SEED =====
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ===== Path =====
src_path = Path("./ASVspoof5/protocol/ASVspoof5.eval.track_2.trial.tsv")
out_path = Path("./PeerJ/protocol/ASVspoof5.csv")

# ===== read file =====
df = pd.read_csv(src_path, sep=r"\s+", header=None, names=["enroll_id", "utt_id", "gender", "label", "trial_type"])
print(f"Loaded {len(df):,} entries")
cnt_spoof = len(df[df["trial_type"] == "spoof"])
cnt_nontarget = len(df[df["trial_type"] == "nontarget"])
print(f"spoof: {cnt_spoof}, nontarget: {cnt_nontarget}")

# ===== spoof/nontarget 33,327 sampling =====
target_count = 33327
spoof_bal = df[df["trial_type"] == "spoof"].sample(n=target_count, random_state=SEED)
nontarget_bal = df[df["trial_type"] == "nontarget"].sample(n=target_count, random_state=SEED)
rest = df[~df["trial_type"].isin(["spoof", "nontarget"])]

# ===== merge =====
balanced_df = pd.concat([spoof_bal, nontarget_bal, rest], ignore_index=True)
print(f"✅ Final balanced count: {len(balanced_df)}")

# ===== save =====
balanced_df = balanced_df.drop(columns=["gender"])
balanced_df.to_csv(out_path, sep=" ", index=False, header=False)
print(f" Saved balanced protocol to: {out_path}")
print(f" Random seed fixed to {SEED}")
